# Multi-Modal Product Classifier

This notebook trains a product category classifier that combines image features from ResNet-18 and text features from BERT.

<h1> I ran the code in Powershell and I have the best_model in the output folder. <h1>

## 1. Setup

Install dependencies before running the notebook:

```bash
pip install -r requirements.txt
```

In [7]:
from pathlib import Path
import json

PROJECT_ROOT = Path(r"c:\Users\Kanan\Downloads\archive (4)")
DATASET_DIR = PROJECT_ROOT / 'fashion-dataset'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
DATASET_DIR.exists(), (DATASET_DIR / 'styles.csv').exists(), (DATASET_DIR / 'images').exists()

(True, True, True)

## 2. Preprocess

This step filters to the top 10 `articleType` labels, drops rows without images, builds product text, and creates stratified splits.

In [12]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [ ]:
from src.preprocess import create_processed_dataset, validate_processed_dataset

paths = create_processed_dataset(DATASET_DIR, PROCESSED_DIR, top_k=10)
validate_processed_dataset(paths['metadata'])

In [ ]:
import pandas as pd

metadata = pd.read_csv(PROCESSED_DIR / 'metadata.csv')
metadata[['id', 'label', 'split', 'text']].head()

## 3. Train

Start with a small sample to validate the pipeline, then remove `--sample-limit` for a full run.

In [ ]:
!python -m src.train --dataset-dir fashion-dataset --processed-dir data/processed --output-dir outputs --epochs 1 --batch-size 8 --sample-limit 25 --fusion concat

## 4. Evaluate

Macro-F1 and class-wise accuracy are written under `outputs/evaluation`.

In [ ]:
!python -m src.evaluate --checkpoint outputs/best_model.pt --processed-dir data/processed --output-dir outputs/evaluation

In [ ]:
metrics = json.loads((OUTPUT_DIR / 'evaluation' / 'metrics.json').read_text())
metrics['macro_f1'], metrics['accuracy'], metrics['class_wise'][:3]

## 5. Bonus: Attention Fusion and Gradio

Train attention fusion with `--fusion attention`, then launch the app from a terminal:

```bash
python app.py --checkpoint outputs/best_model.pt
```